In [3]:
# =========================================================
# IMPORTS
# =========================================================

import numpy as np
import pandas as pd

import mlflow
import mlflow.catboost

import optuna

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mlflow.set_experiment(
    "catboost-uber-demand-prediction"
)
# load the training and test data

train_data_path = "data/train_new.csv"
test_data_path = "data/test_new.csv"

train_df = pd.read_csv(train_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

test_df = pd.read_csv(test_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

2026/05/28 05:07:35 INFO mlflow.tracking.fluent: Experiment with name 'catboost-uber-demand-prediction' does not exist. Creating a new experiment.


In [5]:
import mlflow
import mlflow.catboost
import optuna
import numpy as np
import pandas as pd

from catboost import (
    CatBoostRegressor
)

from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.model_selection import (
    TimeSeriesSplit
)

# ===================================================
# CREATE TIME FEATURES
# ===================================================

df = train_df.copy()

df.index = pd.to_datetime(
    df.index
)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

df["hour"] = df.index.hour

df["day"] = df.index.day

df["month"] = df.index.month

df["day_of_week"] = (
    df.index.day_name()
)

df["is_weekend"] = (
    df.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL FEATURES
# ---------------------------------------------------

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

df["lag_1"] = (
    df["total_pickups"]
    .shift(1)
)

df["lag_24"] = (
    df["total_pickups"]
    .shift(24)
)

df["lag_168"] = (
    df["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# ROLLING FEATURES
# ---------------------------------------------------

df["rolling_mean_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(24)
    .mean()
)

df["rolling_std_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(24)
    .std()
)

df["rolling_mean_168"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(168)
    .mean()
)

df["rolling_std_168"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(168)
    .std()
)

# ---------------------------------------------------
# EXTRA FEATURES
# ---------------------------------------------------

df["is_peak_hour"] = (
    df["hour"].isin(
        [7,8,9,17,18,19]
    )
).astype(int)

df["is_night"] = (
    df["hour"].isin(
        [0,1,2,3,4,5]
    )
).astype(int)

df["weekend_hour_interaction"] = (
    df["hour"] *
    df["is_weekend"]
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

df = df.dropna()

# ===================================================
# TEST FEATURES
# ===================================================

test_processed = test_df.copy()

test_processed.index = pd.to_datetime(
    test_processed.index
)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

test_processed["hour"] = (
    test_processed.index.hour
)

test_processed["day"] = (
    test_processed.index.day
)

test_processed["month"] = (
    test_processed.index.month
)

test_processed["day_of_week"] = (
    test_processed.index.day_name()
)

test_processed["is_weekend"] = (
    test_processed.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL FEATURES
# ---------------------------------------------------

test_processed["hour_sin"] = np.sin(
    2 * np.pi * test_processed["hour"] / 24
)

test_processed["hour_cos"] = np.cos(
    2 * np.pi * test_processed["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

test_processed["lag_1"] = (
    test_processed["total_pickups"]
    .shift(1)
)

test_processed["lag_24"] = (
    test_processed["total_pickups"]
    .shift(24)
)

test_processed["lag_168"] = (
    test_processed["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# ROLLING FEATURES
# ---------------------------------------------------

test_processed["rolling_mean_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(24)
    .mean()
)

test_processed["rolling_std_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(24)
    .std()
)

test_processed["rolling_mean_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(168)
    .mean()
)

test_processed["rolling_std_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(168)
    .std()
)

# ---------------------------------------------------
# EXTRA FEATURES
# ---------------------------------------------------

test_processed["is_peak_hour"] = (
    test_processed["hour"].isin(
        [7,8,9,17,18,19]
    )
).astype(int)

test_processed["is_night"] = (
    test_processed["hour"].isin(
        [0,1,2,3,4,5]
    )
).astype(int)

test_processed["weekend_hour_interaction"] = (
    test_processed["hour"] *
    test_processed["is_weekend"]
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

test_processed = (
    test_processed.dropna()
)

# ===================================================
# FEATURES / TARGET
# ===================================================

X_full = df.drop(
    columns=["total_pickups"]
)

y_full = df["total_pickups"]

X_test = test_processed.drop(
    columns=["total_pickups"]
)

y_test = test_processed[
    "total_pickups"
]

# ===================================================
# CATEGORICAL FEATURES
# ===================================================

categorical_cols = [
    "region",
    "day_of_week"
]

for col in categorical_cols:

    X_full[col] = (
        X_full[col]
        .astype(str)
    )

    X_test[col] = (
        X_test[col]
        .astype(str)
    )

# ===================================================
# CATBOOST CATEGORY INDICES
# ===================================================

cat_features = [

    X_full.columns.get_loc(
        "region"
    ),

    X_full.columns.get_loc(
        "day_of_week"
    )
]

# ===================================================
# WAPE
# ===================================================

def wape(y_true, y_pred):

    return (
        np.sum(
            np.abs(y_true - y_pred)
        )
        /
        np.sum(
            np.abs(y_true)
        )
    ) * 100

# ===================================================
# TIME SERIES CV
# ===================================================

tscv = TimeSeriesSplit(
    n_splits=5
)

# ===================================================
# MLFLOW
# ===================================================

mlflow.set_experiment(
    "catboost_rmse_cv"
)

# ===================================================
# OBJECTIVE
# ===================================================

def objective(trial):

    rmse_scores = []

    params = {

        "iterations": trial.suggest_int(
            "iterations",
            200,
            700,
            step=50
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            1e-2,
            0.1,
            log=True
        ),

        "depth": trial.suggest_int(
            "depth",
            4,
            10
        ),

        "l2_leaf_reg": (
            trial.suggest_float(
                "l2_leaf_reg",
                1e-3,
                10,
                log=True
            )
        ),

        "random_strength": (
            trial.suggest_float(
                "random_strength",
                0.1,
                5
            )
        ),

        "bagging_temperature": (
            trial.suggest_float(
                "bagging_temperature",
                0,
                5
            )
        ),

        "loss_function": "RMSE",

        "eval_metric": "RMSE",

        "random_seed": 42,

        "verbose": False
    }

    # ===============================================
    # CROSS VALIDATION
    # ===============================================

    for train_idx, val_idx in tscv.split(X_full):

        X_train = X_full.iloc[
            train_idx
        ]

        X_val = X_full.iloc[
            val_idx
        ]

        y_train = y_full.iloc[
            train_idx
        ]

        y_val = y_full.iloc[
            val_idx
        ]

        model = CatBoostRegressor(
            **params
        )

        model.fit(

            X_train,
            y_train,

            cat_features=cat_features
        )

        y_pred = model.predict(
            X_val
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                y_pred
            )
        )

        rmse_scores.append(
            rmse
        )

    return np.mean(
        rmse_scores
    )

# ===================================================
# OPTUNA SEARCH
# ===================================================

study = optuna.create_study(
    direction="minimize"
)

study.optimize(

    objective,

    n_trials=50
)

print("Best Params:")

print(study.best_params)

print("\nBest CV RMSE:")

print(study.best_value)

# ===================================================
# FINAL MODEL
# ===================================================

final_model = CatBoostRegressor(

    **study.best_params,

    loss_function="RMSE",

    eval_metric="RMSE",

    random_seed=42,

    verbose=False
)

# ===================================================
# TRAIN FINAL MODEL
# ===================================================

final_model.fit(

    X_full,
    y_full,

    cat_features=cat_features
)

# ===================================================
# TEST PREDICTIONS
# ===================================================

y_test_pred = final_model.predict(
    X_test
)

# ===================================================
# FINAL METRICS
# ===================================================

final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

final_mape = (
    mean_absolute_percentage_error(
        y_test,
        y_test_pred
    )
)

final_wape = wape(
    y_test,
    y_test_pred
)

final_r2 = r2_score(
    y_test,
    y_test_pred
)

# ===================================================
# PRINT RESULTS
# ===================================================

print("\nFINAL RESULTS")

print("MAE :", final_mae)

print("RMSE:", final_rmse)

print("MAPE:", final_mape)

print("WAPE:", final_wape)

print("R2  :", final_r2)

# ===================================================
# LOG FINAL MODEL
# ===================================================

with mlflow.start_run(

    run_name="best_catboost_model"

):

    mlflow.log_params(
        study.best_params
    )

    mlflow.log_metric(
        "FINAL_MAE",
        final_mae
    )

    mlflow.log_metric(
        "FINAL_RMSE",
        final_rmse
    )

    mlflow.log_metric(
        "FINAL_MAPE",
        final_mape
    )

    mlflow.log_metric(
        "FINAL_WAPE",
        final_wape
    )

    mlflow.log_metric(
        "FINAL_R2",
        final_r2
    )

    mlflow.catboost.log_model(
        cb_model=final_model,
        artifact_path="model"
    )

print(
    "\nFinal CatBoost model logged successfully."
)

[I 2026-05-28 05:12:13,339] A new study created in memory with name: no-name-564ab17f-fddb-4e1d-b2c0-b2dd1234f24a
[I 2026-05-28 05:14:21,472] Trial 0 finished with value: 28.667233475973234 and parameters: {'iterations': 600, 'learning_rate': 0.04025328539440913, 'depth': 4, 'l2_leaf_reg': 0.025814381157996984, 'random_strength': 0.2237976497990533, 'bagging_temperature': 4.853303531431265}. Best is trial 0 with value: 28.667233475973234.
[I 2026-05-28 05:16:46,093] Trial 1 finished with value: 31.232877868642497 and parameters: {'iterations': 350, 'learning_rate': 0.0763778923216608, 'depth': 8, 'l2_leaf_reg': 0.0024657885396159518, 'random_strength': 4.455391206043033, 'bagging_temperature': 2.173317997302236}. Best is trial 0 with value: 28.667233475973234.
[I 2026-05-28 05:18:58,219] Trial 2 finished with value: 29.445275500990725 and parameters: {'iterations': 450, 'learning_rate': 0.07526049874718334, 'depth': 7, 'l2_leaf_reg': 6.335158551813312, 'random_strength': 3.491936434762

Best Params:
{'iterations': 600, 'learning_rate': 0.028661854554134652, 'depth': 5, 'l2_leaf_reg': 2.291394676300864, 'random_strength': 0.4661732410358901, 'bagging_temperature': 4.996918421988498}

Best CV RMSE:
28.370628880789035

FINAL RESULTS
MAE : 13.296233847693642
RMSE: 21.56596986110002
MAPE: 343390984081024.3
WAPE: 10.077673914121648
R2  : 0.9727784680097916


2026/05/28 14:55:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Final CatBoost model logged successfully.
